In [ ]:
import pandas as pd
import re
import numpy as np

In [ ]:
df = pd.read_csv("../datasets/orders.csv").copy()

In [ ]:
df.head(5)

,USER_ID,ORDER_ID,PURCHASE_TS,SHIP_TS,REFUND_TS,PRODUCT_NAME,PRODUCT_ID,USD_PRICE,PURCHASE_PLATFORM,MARKETING_CHANNEL,ACCOUNT_CREATION_METHOD,COUNTRY_CODE
0,2c06175e,0001328c3c220830,12/24/2020,12/13/2020,NaN,Nintendo Switch,e682,168.00,website,affiliate,unknown,US
1,ee8e5bc2,0002af7a5c6100772,10/1/2020,9/21/2020,NaN,Nintendo Switch,e682,160.61,website,direct,desktop,DE
2,9eb4efe0,0002b8350e167074,4/21/2020,2/16/2020,NaN,Nintendo Switch,8d0d,151.20,website,direct,desktop,US
3,cac7cbaf,0006d06b98385729,4/7/2020,4/4/2020,4/28/2022,Sony PlayStation 5 Bundle,54ed,1132.82,website,direct,desktop,AU
4,6b0230bc,00097279a2f46150,11/24/2020,8/2/2020,NaN,Nintendo Switch,8d0d,33.89,website,direct,desktop,TR


In [ ]:
# ====================================================================
# STEP 1: HEADER CLEANING
# ====================================================================
print(df.columns.tolist())
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print("headers successfully cleaned")
print(df.columns.tolist())

['USER_ID', 'ORDER_ID', 'PURCHASE_TS', 'SHIP_TS', 'REFUND_TS', 'PRODUCT_NAME', 'PRODUCT_ID', 'USD_PRICE', 'PURCHASE_PLATFORM', 'MARKETING_CHANNEL', 'ACCOUNT_CREATION_METHOD', 'COUNTRY_CODE']
headers successfully cleaned
['user_id', 'order_id', 'purchase_ts', 'ship_ts', 'refund_ts', 'product_name', 'product_id', 'usd_price', 'purchase_platform', 'marketing_channel', 'account_creation_method', 'country_code']


In [ ]:
# ====================================================================
# STEP 2: TYPE COVERSION & CURRENCY CLEANING
# ====================================================================
# columns: unit_price_usd, shipping_cost_usd

unit_price_usd_mask = df["usd_price"].astype(str).str.contains(r"\$")
print(df.loc[unit_price_usd_mask, ["order_id", "usd_price"]].head(3))

Empty DataFrame
Columns: [order_id, usd_price]
Index: []


In [ ]:
df[["order_id","product_id", "usd_price"]]

,order_id,product_id,usd_price
0,0001328c3c220830,e682,168.00
1,0002af7a5c6100772,e682,160.61
2,0002b8350e167074,8d0d,151.20
3,0006d06b98385729,54ed,1132.82
4,00097279a2f46150,8d0d,33.89
...,...,...,...
21859,fff0b6a1e9996384,54ed,1527.06
21860,fff4592dc6d103535,891b,467.88
21861,fff4592dc6d103537,891b,467.88
21862,fff829b061e16171,8d4f,67.07


In [ ]:
df.head(3)

,user_id,order_id,purchase_ts,ship_ts,refund_ts,product_name,product_id,usd_price,purchase_platform,marketing_channel,account_creation_method,country_code
0,2c06175e,0001328c3c220830,12/24/2020,12/13/2020,NaN,Nintendo Switch,e682,168.00,website,affiliate,unknown,US
1,ee8e5bc2,0002af7a5c6100772,10/1/2020,9/21/2020,NaN,Nintendo Switch,e682,160.61,website,direct,desktop,DE
2,9eb4efe0,0002b8350e167074,4/21/2020,2/16/2020,NaN,Nintendo Switch,8d0d,151.20,website,direct,desktop,US


In [ ]:
# ====================================================================
# STEP 3: DATE PARSING
# ====================================================================
# columns: purchase_ts, ship_ts

print(df["purchase_ts"].dtype)
print(df["ship_ts"].dtype)

df["purchase_ts"] = pd.to_datetime(df["purchase_ts"], format ="mixed", errors = "coerce")
df["ship_ts"] = pd.to_datetime(df["ship_ts"], format ="mixed", errors = "coerce")

object
object


In [ ]:
# the nice to have--s
df["purchase_y"] = df["purchase_ts"].dt.to_period("Y")
df["purchase_ym"] = df["purchase_ts"].dt.to_period("M")
df["purchase_ymd"] = df["purchase_ts"].dt.to_period("D")
df["purchase_ymw"] = df["purchase_ts"].dt.to_period("W")
df["purchase_q"] = df["purchase_ts"].dt.to_period("Q")

# df = df.drop(columns=["purchase_year", "purchase_year_month", "purchase_year_month_week", "purchase_quarter"])

In [ ]:
df["ship_y"] = df["ship_ts"].dt.to_period("Y")
df["ship_ym"] = df["ship_ts"].dt.to_period("M")
df["ship_ymd"] = df["ship_ts"].dt.to_period("D")
df["ship_ymw"] = df["ship_ts"].dt.to_period("W")
df["ship_q"] = df["ship_ts"].dt.to_period("Q")

In [ ]:
df["purchase_platform"] = df["purchase_platform"].str.lower()
df["purchase_platform"].unique()

array(['website', 'mobile app'], dtype=object)

In [ ]:
df["marketing_channel"] = df["marketing_channel"].str.lower()
df["marketing_channel"].unique()

array(['affiliate', 'direct', 'email', 'social media', nan, 'unknown'],
      dtype=object)

In [ ]:
df['marketing_channel'] = df['marketing_channel'].replace({np.nan: "organic", "uknown": "organic"})

In [ ]:
df = df[~df["purchase_ts"].isna()]

In [ ]:
df["account_creation_method"].unique()

array(['unknown', 'desktop', 'mobile', 'tablet', nan, 'tv'], dtype=object)

In [ ]:
df["product_name"].unique()

array(['Nintendo Switch', 'Sony PlayStation 5 Bundle',
       '27in 4K gaming monitor', 'JBL Quantum 100 Gaming Headset',
       'Dell Gaming Mouse', 'Acer Nitro V Gaming Laptop',
       'Lenovo IdeaPad Gaming 3', 'Razer Pro Gaming Headset',
       '27inches 4k gaming monitor'], dtype=object)

In [ ]:
df["product_name"] = df["product_name"].replace("27inches 4k gaming monitor", "27in 4K gaming monitor")

In [ ]:
df[df["product_name"] == "27in 4K gaming monitor"]["product_id"].mode()

0    891b
Name: product_id, dtype: object

In [ ]:
# ====================================================================
# FINAL STEP: EXPORTING THE NEWLY CLEANED DATA
# ====================================================================
df.to_csv("../datasets/sales_data_cln.csv", index=False)

In [ ]:
df.head()